# mcp

> Minimal MCP (Model Context Protocol) stdio server for kosha — zero extra dependencies. Exposes search, find_related, node_info, where_to_add, public_api, status and sync as native agent tools for Claude Code, Cursor, Codex, OpenCode and any other MCP client. Speaks JSON-RPC 2.0, one message per line, implementing the tools subset of the protocol.

In [ ]:
#| default_exp mcp

In [ ]:
#| export
import json, sys, os
from contextlib import redirect_stdout
from fastcore.all import Path
from kosha.core import Kosha
from kosha.cli import _to_json

## Tools

> Tool schemas double as agent-facing documentation: the descriptions tell the model when to reach for kosha instead of grep/read.

In [ ]:
#| export
PROTOCOL = '2024-11-05'

def _schema(props, req=None): return dict(type='object', properties=props, required=req or [])

_q = dict(type='string', description='natural-language or symbol query')
_lim = lambda d: dict(type='integer', description=f'max results (default {d})')
TOOLS = [
  dict(name='search',
       description='Hybrid FTS+vector+call-graph search over the repo AND installed packages. Returns code snippets with callers, callees and pagerank. Use before writing new code.',
       inputSchema=_schema(dict(query=_q, limit=_lim(10)), ['query'])),
  dict(name='env_search',
       description='Search only the code of installed packages (check whether a dependency already implements something).',
       inputSchema=_schema(dict(query=_q, limit=_lim(10)), ['query'])),
  dict(name='find_related',
       description='Chunks semantically similar to the code at file:line, plus its call-graph neighborhood.',
       inputSchema=_schema(dict(path=dict(type='string', description='file path, absolute or repo-relative'),
                                line=dict(type='integer', description='line number inside the chunk'),
                                limit=_lim(10)), ['path'])),
  dict(name='node_info',
       description='Callers, callees, co-dispatched peers and pagerank for a fully-qualified node name (e.g. fastcore.basics.merge).',
       inputSchema=_schema(dict(name=dict(type='string')), ['name'])),
  dict(name='where_to_add',
       description='Likely insertion points (file:line and peer functions) for new code matching a description.',
       inputSchema=_schema(dict(description=dict(type='string'), limit=_lim(5)), ['description'])),
  dict(name='public_api',
       description='Public API entries of a package (respects __all__ and fastcore @patch).',
       inputSchema=_schema(dict(pkg=dict(type='string')), ['pkg'])),
  dict(name='status',
       description='Index freshness: file/package/graph-node counts, stale files and stale packages.',
       inputSchema=_schema({})),
  dict(name='sync',
       description='Incrementally re-index the repo, packages and call graph. Run when status reports stale entries.',
       inputSchema=_schema({})),
]

def _call(k, name, a):
	'Dispatch one MCP tool call to the Kosha API.'
	if name == 'search':        return k.context(a['query'], limit=a.get('limit', 10))
	if name == 'env_search':    return k.env_context(a['query'], limit=a.get('limit', 10))
	if name == 'find_related':  return k.find_related(a['path'], line=a.get('line'), limit=a.get('limit', 10))
	if name == 'node_info':     return k.ni(a['name'])
	if name == 'where_to_add':  return k.where_to_add(a['description'], limit=a.get('limit', 5))
	if name == 'public_api':    return k.public_api(a['pkg'])
	if name == 'status':        return k.status()
	if name == 'sync':          return (k.sync(), 'synced')[1]
	raise ValueError(f'unknown tool: {name}')

## Server loop

> stdout belongs to the protocol, so any print from kosha (progress bars, sync messages) is redirected to stderr during tool calls. The Kosha instance is created lazily on the first call so `initialize`/`tools/list` stay instant.

In [ ]:
#| export
def _res(rid, result): return dict(jsonrpc='2.0', id=rid, result=result)

def handle(req:dict, state:dict) -> dict | None:
	'Handle one JSON-RPC request. Returns a response dict, or None for notifications.'
	rid, m, p = req.get('id'), req.get('method', ''), req.get('params') or {}
	if m == 'initialize':
		try: ver = __import__('importlib.metadata', fromlist=['version']).version('koshas')
		except Exception: ver = '0'
		return _res(rid, dict(protocolVersion=p.get('protocolVersion') or PROTOCOL,
		                      capabilities=dict(tools={}), serverInfo=dict(name='kosha', version=ver)))
	if m.startswith('notifications/'): return None
	if m == 'ping': return _res(rid, {})
	if m == 'tools/list': return _res(rid, dict(tools=TOOLS))
	if m == 'tools/call':
		try:
			# kosha prints progress to stdout; MCP owns stdout, so route prints to stderr
			with redirect_stdout(sys.stderr):
				if state.get('k') is None: state['k'] = Kosha()
				out = _call(state['k'], p.get('name'), p.get('arguments') or {})
			return _res(rid, dict(content=[dict(type='text', text=json.dumps(_to_json(out), default=str))]))
		except Exception as e:
			return _res(rid, dict(content=[dict(type='text', text=f'error: {e}')], isError=True))
	if rid is None: return None
	return dict(jsonrpc='2.0', id=rid, error=dict(code=-32601, message=f'method not found: {m}'))

def serve():
	'Blocking MCP stdio loop: one JSON-RPC message per line on stdin/stdout.'
	state = {}
	for line in sys.stdin:
		if not (line := line.strip()): continue
		try: req = json.loads(line)
		except Exception: continue
		if (resp := handle(req, state)) is not None:
			sys.stdout.write(json.dumps(resp) + '\n')
			sys.stdout.flush()

In [ ]:
# protocol smoke test, no index needed
r = handle(dict(jsonrpc='2.0', id=1, method='initialize', params={}), {})
assert r['result']['serverInfo']['name'] == 'kosha' and r['result']['capabilities'] == dict(tools={})
assert handle(dict(jsonrpc='2.0', method='notifications/initialized'), {}) is None
r = handle(dict(jsonrpc='2.0', id=2, method='tools/list'), {})
assert {t['name'] for t in r['result']['tools']} >= {'search','env_search','find_related','node_info','where_to_add','public_api','status','sync'}
r = handle(dict(jsonrpc='2.0', id=3, method='nope'), {})
assert r['error']['code'] == -32601


## Install

> `kosha install` writes project-scoped MCP config for Claude Code and Cursor, and prints one-liners for Codex and OpenCode.

In [ ]:
#| export
SERVER_CFG = dict(command='kosha', args=['mcp'])

def _merge_json(p:Path, key_path:list, value) -> Path:
	'Merge value into the JSON file at a nested key path, preserving unrelated content.'
	d = json.loads(p.read_text()) if p.exists() else {}
	cur = d
	for k in key_path[:-1]: cur = cur.setdefault(k, {})
	cur[key_path[-1]] = value
	p.parent.mkdir(parents=True, exist_ok=True)
	p.write_text(json.dumps(d, indent=2) + '\n')
	return p

def install_mcp(root=None) -> list:
	'Write kosha MCP server config for Claude Code (.mcp.json) and Cursor (.cursor/mcp.json); print setup hints for other agents.'
	root = Path(root or '.')
	written = [_merge_json(root/'.mcp.json', ['mcpServers','kosha'], SERVER_CFG),
	           _merge_json(root/'.cursor/mcp.json', ['mcpServers','kosha'], SERVER_CFG)]
	print(f'MCP config installed → {list(map(str, written))}')
	print('Codex:    codex mcp add kosha -- kosha mcp')
	print('OpenCode: add {"mcp": {"kosha": {"type": "local", "command": ["kosha", "mcp"]}}} to opencode.json')
	return written